# Build ABO Physics Subset

Этот ноутбук пересобирает подмножество Amazon ABO для задачи извлечения физических свойств.

Выход:
- `dataset/abo_physics_val/meta.json`
- `dataset/abo_physics_val/images/...`
- `dataset/abo_physics_val/summary.json`


In [1]:
# !pip install -q boto3 rembg onnxruntime


In [2]:
import subprocess

# Настройки генерации
LISTING_SHARDS = "0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15"
MAX_SAMPLES = 50
MIN_KNOWN_PROPERTIES = 4
MAX_PER_PRODUCT_TYPE = 25
EXCLUDE_PRODUCT_TYPES = "CELLULAR_PHONE_CASE,PORTABLE_ELECTRONIC_DEVICE_COVER"
SEED = 42
DOWNLOAD_MISSING = True  # Для Colab обычно True
GENERATE_MASKS = True
MASK_BACKEND = "rembg"  # rembg | simple

cmd = [
    "python", "scripts/build_abo_physics_subset.py",
    "--dataset-root", "dataset",
    "--cache-dir", "dataset/abo_vlm_val/_cache",
    "--source-images-dir", "dataset/abo_vlm_val/images",
    "--listing-shards", LISTING_SHARDS,
    "--max-samples", str(MAX_SAMPLES),
    "--min-known-properties", str(MIN_KNOWN_PROPERTIES),
    "--max-per-product-type", str(MAX_PER_PRODUCT_TYPE),
    "--exclude-product-types", EXCLUDE_PRODUCT_TYPES,
    "--seed", str(SEED),
    "--clear-output",
]

if DOWNLOAD_MISSING:
    cmd.append("--download-missing")
if GENERATE_MASKS:
    cmd += [
        "--generate-masks",
        "--mask-backend", MASK_BACKEND,
        "--min-mask-area-ratio", "0.01",
        "--max-mask-area-ratio", "0.95",
    ]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


Running: python scripts/build_abo_physics_subset.py --dataset-root dataset --cache-dir dataset/abo_vlm_val/_cache --source-images-dir dataset/abo_vlm_val/images --listing-shards 0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15 --max-samples 50 --min-known-properties 4 --max-per-product-type 25 --exclude-product-types CELLULAR_PHONE_CASE,PORTABLE_ELECTRONIC_DEVICE_COVER --seed 42 --clear-output --download-missing --generate-masks --mask-backend rembg --min-mask-area-ratio 0.01 --max-mask-area-ratio 0.95
Wrote: dataset/abo_physics_val/meta.json
Wrote: dataset/abo_physics_val/summary.json
Samples: 50


CompletedProcess(args=['python', 'scripts/build_abo_physics_subset.py', '--dataset-root', 'dataset', '--cache-dir', 'dataset/abo_vlm_val/_cache', '--source-images-dir', 'dataset/abo_vlm_val/images', '--listing-shards', '0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15', '--max-samples', '50', '--min-known-properties', '4', '--max-per-product-type', '25', '--exclude-product-types', 'CELLULAR_PHONE_CASE,PORTABLE_ELECTRONIC_DEVICE_COVER', '--seed', '42', '--clear-output', '--download-missing', '--generate-masks', '--mask-backend', 'rembg', '--min-mask-area-ratio', '0.01', '--max-mask-area-ratio', '0.95'], returncode=0)

In [3]:
import json
from pathlib import Path

meta_path = Path("dataset/abo_physics_val/meta.json")
summary_path = Path("dataset/abo_physics_val/summary.json")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("Samples:", len(meta))
print("\nProperty distributions:")
for k, v in summary.get("property_distributions", {}).items():
    print(f"- {k}: {v}")

if meta:
    print("\nFirst sample:")
    print(json.dumps(meta[0], ensure_ascii=False, indent=2)[:1200])


Samples: 50

Property distributions:
- material: {'metal': 7, 'rubber': 3, 'plastic': 6, 'fabric': 18, 'stone': 10, 'paper': 3, 'wood': 3}
- rigidity: {'rigid': 25, 'flexible': 7, 'soft': 18}
- transparency: {'opaque': 50}
- surface: {'smooth': 13, 'fuzzy': 9, 'mixed': 16, 'rough': 12}
- fragility: {'durable': 46, 'fragile': 4}

First sample:
{
  "image_id": "51K8AWLcDLL",
  "path": "abo_physics_val/images/c3/c38d1143.jpg",
  "primary_object": "fineearring",
  "properties": {
    "material": "metal",
    "rigidity": "rigid",
    "transparency": "opaque",
    "surface": "smooth",
    "fragility": "durable"
  },
  "notes": "metadata-derived physical pseudo-label",
  "abo_meta": {
    "item_id": "B01F7B2M0C",
    "product_type": "FINEEARRING",
    "domain_name": "amazon.com.mx",
    "title": "Amazon CollectionRhodium Plated Sterling Silver Cushion Cut Created Emerald 5mm and Created White Sapphire Halo Stud Earrings"
  },
  "mask_path": "abo_physics_val/masks/c3/c38d1143.png",
  "mask_sou